# Lightweight ML Text Classification Experiment

**Paper:** A Comparative Study of Classical Lightweight Machine Learning Models for Text Classification

**Runtime:** CPU is sufficient (no GPU required). Optional T4 is not needed for this benchmark.

**Steps:**
1. Run all cells in order (Runtime → Run all)
2. Results save to `/content/results/`
3. Copy printed LaTeX tables into `manuscript.tex`
4. Download figures from `/content/results/figures/`

In [ ]:
!pip -q install xgboost

In [ ]:
import os
import re
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ============ CONFIG (matches manuscript) ============
RANDOM_SEED = 42
TEST_SIZE = 0.2
N_RUNS = 3
AG_NEWS_SUBSET = 20000
MAX_FEATURES = 20000
NGRAM_RANGE = (1, 2)
MIN_DF = 2

RESULTS_DIR = Path('/content/results')
FIG_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

MODELS = {
    'Logistic Regression': LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=RANDOM_SEED),
    'Linear SVM': LinearSVC(C=1.0, random_state=RANDOM_SEED),
    'Naive Bayes': MultinomialNB(alpha=1.0),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=RANDOM_SEED,
                             eval_metric='mlogloss', verbosity=0, n_jobs=-1)
}

print('Config loaded. Results dir:', RESULTS_DIR)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_sms_spam():
    import io
    import zipfile
    import urllib.request

    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
    with urllib.request.urlopen(url) as response:
        with zipfile.ZipFile(io.BytesIO(response.read())) as zf:
            with zf.open('SMSSpamCollection') as f:
                df = pd.read_csv(
                    f, sep='\t', header=None, names=['label', 'text'], encoding='latin-1'
                )
    df['label'] = df['label'].map({'ham': 0, 'spam': 1})
    df['text'] = df['text'].apply(clean_text)
    return df


def load_ag_news(subset=AG_NEWS_SUBSET):
    """Load AG News from HuggingFace JSONL (no datasets library required)."""
    import json
    import urllib.request

    base = 'https://huggingface.co/datasets/sh0416/ag_news/resolve/main'
    rows = []
    for split in ('train', 'test'):
        with urllib.request.urlopen(f'{base}/{split}.jsonl') as resp:
            for line in resp:
                item = json.loads(line)
                rows.append({
                    'label': item['label'] - 1,
                    'text': clean_text(f"{item['title']} {item['description']}"),
                })
    df = pd.DataFrame(rows)

    if subset and subset < len(df):
        per_class = subset // df['label'].nunique()
        df = (
            df.groupby('label', group_keys=False)
            .sample(n=per_class, random_state=RANDOM_SEED)
            .reset_index(drop=True)
        )
    return df


sms_df = load_sms_spam()
ag_df = load_ag_news()

print('SMS Spam:', sms_df.shape, sms_df['label'].value_counts().to_dict())
print('AG News:', ag_df.shape, ag_df['label'].value_counts().to_dict())

dataset_summary = pd.DataFrame([
    {'Dataset': 'SMS Spam', 'Classes': 2, 'Total': len(sms_df),
     'Train': int(len(sms_df)*(1-TEST_SIZE)), 'Test': int(len(sms_df)*TEST_SIZE)},
    {'Dataset': 'AG News', 'Classes': 4, 'Total': len(ag_df),
     'Train': int(len(ag_df)*(1-TEST_SIZE)), 'Test': int(len(ag_df)*TEST_SIZE)},
])
dataset_summary

In [ ]:
def run_benchmark(df, dataset_name, average='binary'):
    rows = []
    all_runs = []
    best_f1 = -1
    best_model_name = None
    best_y_true = None
    best_y_pred = None

    X = df['text'].values
    y = df['label'].values

    for model_name, clf in MODELS.items():
        run_metrics = []
        for run in range(N_RUNS):
            seed = RANDOM_SEED + run
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
            )

            pipe = Pipeline([
                ('tfidf', TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE,
                                          min_df=MIN_DF, stop_words='english')),
                ('clf', clf)
            ])

            t0 = time.perf_counter()
            pipe.fit(X_train, y_train)
            train_time = time.perf_counter() - t0

            t1 = time.perf_counter()
            y_pred = pipe.predict(X_test)
            infer_total = time.perf_counter() - t1
            infer_ms_per_1k = (infer_total / len(X_test)) * 1000

            model_path = RESULTS_DIR / f'{dataset_name}_{model_name.replace(" ", "_")}.joblib'
            joblib.dump(pipe, model_path)
            size_mb = model_path.stat().st_size / (1024 * 1024)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average=average, zero_division=0)
            rec = recall_score(y_test, y_pred, average=average, zero_division=0)
            f1 = f1_score(y_test, y_pred, average=average, zero_division=0)
            eff = f1 / np.log1p(infer_ms_per_1k)

            run_metrics.append({
                'Dataset': dataset_name, 'Model': model_name, 'Run': run + 1,
                'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1,
                'Train_s': train_time, 'Infer_ms_per_1k': infer_ms_per_1k,
                'Size_MB': size_mb, 'Efficiency_E': eff
            })
            all_runs.append(run_metrics[-1])

            if run == 0 and f1 > best_f1:
                best_f1 = f1
                best_model_name = model_name
                best_y_true = y_test
                best_y_pred = y_pred

        avg = pd.DataFrame(run_metrics).mean(numeric_only=True)
        rows.append({
            'Dataset': dataset_name, 'Model': model_name,
            'Accuracy': avg['Accuracy'], 'Precision': avg['Precision'],
            'Recall': avg['Recall'], 'F1': avg['F1'],
            'Train_s': avg['Train_s'], 'Infer_ms_per_1k': avg['Infer_ms_per_1k'],
            'Size_MB': avg['Size_MB'], 'Efficiency_E': avg['Efficiency_E']
        })

    results = pd.DataFrame(rows)
    return results, best_model_name, best_y_true, best_y_pred, pd.DataFrame(all_runs)


print('Running SMS Spam benchmark...')
sms_results, sms_best, sms_y_true, sms_y_pred, sms_runs = run_benchmark(sms_df, 'SMS Spam', average='binary')

print('Running AG News benchmark...')
ag_results, ag_best, ag_y_true, ag_y_pred, ag_runs = run_benchmark(ag_df, 'AG News', average='macro')

all_results = pd.concat([sms_results, ag_results], ignore_index=True)
all_runs = pd.concat([sms_runs, ag_runs], ignore_index=True)
all_results.to_csv(RESULTS_DIR / 'all_results.csv', index=False)
all_runs.to_csv(RESULTS_DIR / 'all_runs.csv', index=False)
print('\nDone. Best SMS model:', sms_best, '| Best AG model:', ag_best)
all_results

In [ ]:
def pct(x):
    return f'{100*x:.1f}'


def print_classification_table(results, dataset_name):
    sub = results[results['Dataset'] == dataset_name]
    print(f'\n=== Classification Table: {dataset_name} ===')
    for _, r in sub.iterrows():
        print(
            f" & {r['Model']} & {pct(r['Accuracy'])} & {pct(r['Precision'])} & "
            f"{pct(r['Recall'])} & {pct(r['F1'])} \\\\"
        )


def print_efficiency_table(results, dataset_name):
    sub = results[results['Dataset'] == dataset_name]
    print(f'\n=== Efficiency Table: {dataset_name} ===')
    for _, r in sub.iterrows():
        print(
            f" & {r['Model']} & {r['Train_s']:.1f} & {r['Infer_ms_per_1k']:.1f} & "
            f"{r['Size_MB']:.2f} & {r['Efficiency_E']:.2f} \\\\"
        )


print_classification_table(all_results, 'SMS Spam')
print_classification_table(all_results, 'AG News')
print_efficiency_table(sms_results, 'SMS Spam')
print_efficiency_table(ag_results, 'AG News')

best_sms = sms_results.loc[sms_results['F1'].idxmax()]
best_ag = ag_results.loc[ag_results['F1'].idxmax()]
fastest = all_results.loc[all_results['Infer_ms_per_1k'].idxmin()]
smallest = all_results.loc[all_results['Size_MB'].idxmin()]

summary = {
    'BEST_MODEL_SMS': best_sms['Model'],
    'BEST_F1_SMS': pct(best_sms['F1']),
    'BEST_MODEL_AG': best_ag['Model'],
    'BEST_F1_AG': pct(best_ag['F1']),
    'FASTEST_MODEL': fastest['Model'],
    'FASTEST_LATENCY_MS': f"{fastest['Infer_ms_per_1k']:.1f}",
    'SMALLEST_MODEL': smallest['Model'],
    'TOTAL_TRAIN_MINUTES': f"{(sms_results['Train_s'].sum() + ag_results['Train_s'].sum())/60:.1f}"
}
print('\n=== MANUSCRIPT PLACEHOLDER VALUES ===')
for k, v in summary.items():
    print(f'{k}: {v}')

with open(RESULTS_DIR / 'manuscript_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

In [ ]:
# Figure 1: Pipeline diagram
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

steps = ['Raw Text', 'Cleaning', 'TF-IDF', 'Classifier', 'Metrics']
fig, ax = plt.subplots(figsize=(9.5, 1.85))
ax.set_xlim(0, 10); ax.set_ylim(0, 1); ax.axis('off')
fig.patch.set_facecolor('white')
edge, text, arrow = '#1a365d', '#1a202c', '#4a5568'
box_w, box_h, y, gap = 1.42, 0.42, 0.42, 0.55
start_x = (10 - (len(steps) * box_w + (len(steps) - 1) * gap)) / 2
centers = []
for i, step in enumerate(steps):
    x = start_x + i * (box_w + gap)
    ax.add_patch(FancyBboxPatch((x, y - box_h / 2), box_w, box_h,
                                boxstyle='round,pad=0.01,rounding_size=0.04',
                                linewidth=1.25, edgecolor=edge, facecolor='white', zorder=2))
    cx = x + box_w / 2
    centers.append(cx)
    ax.text(cx, y, step, ha='center', va='center', fontsize=10, color=text, zorder=3)
for i in range(len(centers) - 1):
    ax.add_patch(FancyArrowPatch((centers[i] + box_w / 2 + 0.05, y),
                                 (centers[i + 1] - box_w / 2 - 0.05, y),
                                 arrowstyle='-|>', mutation_scale=11, linewidth=1.2,
                                 color=arrow, zorder=1))
ax.text(5.0, 0.82, 'Experimental Pipeline', ha='center', va='center', fontsize=11,
        fontweight='bold', color=text)
plt.savefig(FIG_DIR / 'fig1_pipeline.png', dpi=300, bbox_inches='tight', pad_inches=0.12, facecolor='white')
plt.show()

# Figure 2: F1 vs Latency scatter
fig, ax = plt.subplots(figsize=(8, 6))
markers = {'SMS Spam': 'o', 'AG News': 's'}
for dataset, marker in markers.items():
    sub = all_results[all_results['Dataset'] == dataset]
    ax.scatter(sub['Infer_ms_per_1k'], sub['F1'], label=dataset, s=100, marker=marker)
    for _, r in sub.iterrows():
        ax.annotate(r['Model'], (r['Infer_ms_per_1k'], r['F1']), fontsize=8, xytext=(4,4), textcoords='offset points')
ax.set_xlabel('Inference Latency (ms per 1,000 samples)')
ax.set_ylabel('F1-score')
ax.set_title('Accuracy-Efficiency Trade-off')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_f1_latency.png', dpi=200, bbox_inches='tight')
plt.show()

# Figure 3: Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sms_labels = ['ham', 'spam']
ag_labels = ['World', 'Sports', 'Business', 'Sci/Tech']
sns.heatmap(confusion_matrix(sms_y_true, sms_y_pred), annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=sms_labels, yticklabels=sms_labels)
axes[0].set_title(f'SMS Spam ({sms_best})')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
sns.heatmap(confusion_matrix(ag_y_true, ag_y_pred), annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=ag_labels, yticklabels=ag_labels)
axes[1].set_title(f'AG News ({ag_best})')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_confusion.png', dpi=200, bbox_inches='tight')
plt.show()

# Figure 4: Training time bar chart
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = all_results.melt(id_vars=['Dataset', 'Model'], value_vars=['Train_s'], var_name='Metric', value_name='Seconds')
sns.barplot(data=all_results, x='Model', y='Train_s', hue='Dataset', ax=ax)
ax.set_ylabel('Training Time (seconds)')
ax.set_title('Training Time by Model and Dataset')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_training_time.png', dpi=200, bbox_inches='tight')
plt.show()

print('Figures saved to', FIG_DIR)
print('Download results:')
print('  from google.colab import files')
print('  !zip -r results.zip /content/results')
print('  files.download("results.zip")')

In [ ]:
# Optional: zip and download all results
from google.colab import files
!zip -r /content/results.zip /content/results
files.download('/content/results.zip')